# Импорты

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.tensorboard import SummaryWriter
import timm
from timm.data.mixup import Mixup
from timm.loss import SoftTargetCrossEntropy
import os
import random
from collections import defaultdict, Counter
from glob import glob
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
from PIL import Image
import seaborn as sns

# Сиды

In [ ]:
# важно - зафиксировать все сиды
SEED = 9999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# Классы

In [ ]:
class_to_idx = { "Апельсин": 0,
                 "Бананы": 1,
                 "Груши": 2,
                 "Кабачки": 3,
                 "Капуста": 4,
                 "Картофель": 5,
                 "Киви": 6,
                 "Лимон": 7,
                 "Лук": 8,
                 "Мандарины": 9,
                 "Морковь": 10,
                 "Огурцы": 11,
                 "Томаты": 12,
                 "Яблоки зеленые": 13,
                 "Яблоки красные": 14 }

# Инкапсуляция

In [ ]:
class MyDataset(Dataset):
    def __init__(self, images_filepaths, name2label, transform=None):
        self.images_filepaths = images_filepaths
        self.transform = transform
        self.name2label = name2label

    def __len__(self):
        return len(self.images_filepaths)

    def __getitem__(self, idx):
        image_filepath = self.images_filepaths[idx]
        image = cv2.imdecode(np.fromfile(image_filepath, dtype=np.uint8), cv2.IMREAD_UNCHANGED)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.name2label[os.path.normpath(image_filepath).split(os.sep)[-3]]
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return image, label


# Пути COLAB/PC



In [ ]:
# Подгрузка данных с диска

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Перенос данных с диска на локальную среду выполнения
!cp "/content/drive/My Drive/ML/lab1ImageClassification/train.zip" /content/
!cp "/content/drive/My Drive/ML/lab1ImageClassification/test_images.zip" /content/
!cp "/content/drive/My Drive/ML/lab1ImageClassification/sample_submission.csv" /content/
!cp "/content/drive/My Drive/ML/lab1ImageClassification/submission.csv" /content/

In [ ]:
# Разархивировать данные
!unzip -q /content/train.zip -d /content/
!unzip -q /content/test_images.zip -d /content/

In [ ]:
# ПК пути
# dataset_path = 'train/train'
# test_images_dir = "test_images/test_images"
# submission_path = "sample_submission.csv"
# output_path = "submission.csv"

dataset_path = '/content/train/train'
test_images_dir = "/content/test_images/test_images"
submission_path = "/content/sample_submission.csv"
output_path = "/content/submission.csv"

# Загрузка данных
### Учитываем то, что в каждом классе есть подкласс. И их размерности разные.

In [ ]:
def collect_subclass_data(root_path):
    all_files = []
    # Это метка для StratifiedKFold (уникальная для каждого подкласса)
    subclass_labels_for_split = []

    # Счетчик для уникальных ID подклассов
    subclass_id_counter = 0

    # Проходим по Главным классам
    for class_name in sorted(os.listdir(root_path)):
        class_path = os.path.join(root_path, class_name)
        if not os.path.isdir(class_path):
            continue

        # Проходим по Подклассам
        for subclass_name in sorted(os.listdir(class_path)):
            subclass_path = os.path.join(class_path, subclass_name)
            if not os.path.isdir(subclass_path):
                continue

            images = glob(os.path.join(subclass_path, '*.jpg')) + \
                     glob(os.path.join(subclass_path, '*.png')) + \
                     glob(os.path.join(subclass_path, '*.jpeg'))

            if len(images) == 0:
                continue

            images = sorted(images) # сортировка для запуска на разных машинах.

            # Добавляем пути
            all_files.extend(images)

            # Всем картинкам из этой папки присваиваем один и тот же ID подкласса
            # Например, все "Томаты/Черри" будут иметь ID 55
            # А "Томаты/Красные" будут иметь ID 56
            subclass_labels_for_split.extend([subclass_id_counter] * len(images))

            subclass_id_counter += 1

    return np.array(all_files), np.array(subclass_labels_for_split)

# Распределение данных

In [ ]:
all_files, all_labels = collect_subclass_data(dataset_path)

In [ ]:
def count_classes(image_paths, class_to_idx):
    counts = Counter()
    for im_path in image_paths:
        class_name = os.path.normpath(im_path).split(os.sep)[-3]
        class_idx = class_to_idx[class_name]
        counts[class_idx] += 1
    return counts

data_counts = count_classes(all_files, class_to_idx)

idx2class = {v: k for k, v in class_to_idx.items()} # Конвертация class_to_idx -> idx2class

classes = list(idx2class.keys())
class_names = [idx2class[i] for i in classes]

data_values = [data_counts.get(i, 0) for i in classes]

x = np.arange(len(classes))

plt.figure(figsize=(8, 4))
plt.bar(x, data_values, label='Train')

plt.xticks(x, class_names, rotation=45, ha='right')
plt.ylabel('Количество изображений')
plt.title('Распределение классов')
plt.legend()
plt.tight_layout()
plt.show()

# Размеры фотографий (статистика)

In [ ]:
def analyze_image_sizes(file_paths):
    widths = []
    heights = []
    areas = []
    aspect_ratios = []

    print("Анализ размеров изображений...")

    # Проходим по всем файлам
    for file_path in tqdm(file_paths):
        try:
            # Image.open открывает только заголовок (ленивая загрузка), это очень быстро
            with Image.open(file_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                areas.append(w * h)
                aspect_ratios.append(w / h)
        except Exception as e:
            print(f"Ошибка при чтении файла {file_path}: {e}")

    # Превращаем в numpy массивы для быстрых расчетов
    widths = np.array(widths)
    heights = np.array(heights)
    areas = np.array(areas)
    aspect_ratios = np.array(aspect_ratios)
    mean_area = np.mean(areas)
    p95_area = np.percentile(areas, 95)

    print(f"Количество изображений: {len(areas)}")
    print(f"Средняя площадь: {mean_area:.0f} пикс. (~{mean_area/1e6:.2f} MP)")
    print(f"95-й процентиль площади: {p95_area:.0f} пикс. (~{p95_area/1e6:.2f} MP)")
    print(f"Мин. площадь: {np.min(areas)} пикс.")
    print(f"Макс. площадь: {np.max(areas)} пикс.")

    # Размеры (Ширина и Высота)
    print("-" * 20)
    print(f"Ширина (Width):  Средняя={np.mean(widths):.0f}, 95%={np.percentile(widths, 95):.0f}, Макс={np.max(widths)}")
    print(f"Высота (Height): Средняя={np.mean(heights):.0f}, 95%={np.percentile(heights, 95):.0f}, Макс={np.max(heights)}")

    # Соотношение сторон
    print("-" * 20)
    print(f"Соотношение сторон (W/H): Среднее={np.mean(aspect_ratios):.2f}")

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Scatter Plot (Ширина vs Высота)
    sns.scatterplot(x=widths, y=heights, alpha=0.3, ax=axes[0], color='blue')
    axes[0].set_title('Ширина vs Высота')
    axes[0].set_xlabel('Ширина')
    axes[0].set_ylabel('Высота')
    axes[0].scatter([300], [300], color='red', s=100, label='Model Input (300x300)', zorder=5)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # 2. Гистограмма Площади
    sns.histplot(areas, bins=50, kde=True, ax=axes[1], color='green')
    axes[1].axvline(p95_area, color='red', linestyle='--', label=f'95% ({p95_area:.0e})')
    axes[1].set_title('Распределение площадей (Area)')
    axes[1].legend()

    # 3. Гистограмма Соотношения сторон
    sns.histplot(aspect_ratios, bins=50, kde=True, ax=axes[2], color='purple')
    axes[2].axvline(1.0, color='red', linestyle='--', label='Квадрат (1:1)')
    axes[2].set_title('Соотношение сторон (Width / Height)')
    axes[2].set_xlabel('< 1 (Вертикальные) ... 1 (Квадрат) ... > 1 (Горизонтальные)')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

# Запускаем анализ
analyze_image_sizes(all_files)

# Аугументация

In [ ]:
IMG_SIZE = 300

train_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(
        min_height=IMG_SIZE,
        min_width=IMG_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=1.0
    ),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=5,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=0.5
    ),

    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    A.OneOf([
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        A.CLAHE(p=0.1),
    ], p=0.85),

    # Освещение и цвет
    A.OneOf([
        A.ColorJitter(brightness=0.1, contrast=0.0, saturation=0.0, hue=0.0),
        A.ColorJitter(brightness=0.0, contrast=0.1, saturation=0.0, hue=0.0),
        A.ColorJitter(brightness=0.0, contrast=0.0, saturation=0.1, hue=0.0),
    ], p=1),

    # Искажения
    A.OneOf([
        A.GaussianBlur(blur_limit=(3), sigma_limit=(0.1, 1.0), p=0.5),
        A.GaussNoise(std_range=(0.01, 0.02), noise_scale_factor=0.7, p=0.9),
    ], p=9),

    A.CoarseDropout(
        num_holes_range=(1, 16),
        hole_height_range=(0.02, 0.1),
        hole_width_range=(0.02, 0.1),
        fill=0,
        p=0.75
    ),

    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    A.ToTensorV2(),
])

val_transforms = A.Compose([
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(
        min_height=IMG_SIZE,
        min_width=IMG_SIZE,
        border_mode=cv2.BORDER_CONSTANT,
        value=0,
        p=1.0
    ),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2(),
])

In [ ]:
def visualize_augmentations(dataset, idx=None, n_samples=6):
    if idx is None:
        idx = random.randint(0, len(dataset) - 1)

    # Достаем путь к файлу из нашего списка путей
    image_path = dataset.images_filepaths[idx]

    # Читаем оригинал
    image = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    viz_transforms = A.Compose([
        t for t in train_transforms.transforms
        if not isinstance(t, (A.Normalize, A.pytorch.ToTensorV2))
    ])

    # 3. Рисуем сетку
    plt.figure(figsize=(15, 10))

    # Показываем оригинал первым
    plt.subplot(2, 3, 1)
    plt.imshow(image)
    plt.title("Оригинал")
    plt.axis('off')

    # Применяем аугментацию n-1 раз
    for i in range(2, n_samples + 1):
        augmented = viz_transforms(image=image)["image"]

        plt.subplot(2, 3, i)
        plt.imshow(augmented)
        plt.title(f"Вариант {i-1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# --- ЗАПУСК ---
# Передай свой train_dataset в функцию
train_visual_augmentations = MyDataset(images_filepaths=all_files, name2label=class_to_idx, transform=train_transforms)
visualize_augmentations(train_visual_augmentations)

# DEVICE

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Валидация


In [ ]:
@torch.no_grad() # при вызове функции отключаем калькулятор градиентов
def evaluate(model, dataloader, loss_fn, device, desc="Val"):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    # Списки для накопления всех ответов
    all_preds = []
    all_targets = []

    pbar = tqdm(dataloader, desc=desc, leave=False)
    for X_batch, y_batch in pbar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)
        loss = loss_fn(logits, y_batch)

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size

        y_pred = logits.argmax(dim=1)
        total_correct += (y_pred == y_batch).sum().item()
        total_samples += batch_size

        avg_loss = total_loss / max(total_samples, 1)
        acc = total_correct / max(total_samples, 1)

        # Обновляем прогресс-бар (показываем acc, так как f1 на лету не посчитать)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{acc:.4f}")

        # Сохраняем предсказания и реальные метки в список
        all_preds.extend(y_pred.cpu().numpy())
        all_targets.extend(y_batch.cpu().numpy())

    # Финальные метрики
    avg_loss = total_loss / max(total_samples, 1)
    accuracy = total_correct / max(total_samples, 1)

    # Считаем F1 по всему датасету разом
    f1 = f1_score(all_targets, all_preds, average='macro')

    return accuracy, avg_loss, f1

# Функция train

In [ ]:
def train(model, loss_fn, optimizer, train_loader, val_loader, device, weights,\
          writer=None, n_epoch=3, lr_eta=1e-6, ls=0.1):
    best_val_f1 = 0.0
    best_val_loss = 100
    num_iter = 0
    counter_early_stop = 0
    save_path = f"checkpoints/best_model.pth"
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epoch, eta_min=lr_eta) # планировщик скорости обучения
    loss_fn_val = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=ls)
    mixup_fn = Mixup(
        mixup_alpha=0.4,
        cutmix_alpha=1.0,
        prob=1.0,
        switch_prob=0.5,
        label_smoothing=0.1,
        num_classes=15
    )

    for epoch in range(1, n_epoch + 1):
        model.train()

        total_loss = 0.0
        total_correct = 0
        total_samples = 0

        pbar = tqdm(train_loader, desc=f"Ep {epoch}/{n_epoch}", leave=True, dynamic_ncols=True)

        for X_batch, y_batch in pbar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            X_batch, y_batch = mixup_fn(X_batch, y_batch)

            logits = model(X_batch)
            loss = loss_fn(logits, y_batch)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            # накопим метрики для прогресс-бара
            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

            y_true_cls = y_batch.argmax(dim=1)
            y_pred_cls = logits.argmax(dim=1)

            correct = (y_pred_cls == y_true_cls).sum().item()
            total_correct += correct

            avg_loss = total_loss / max(total_samples, 1)
            acc = total_correct / max(total_samples, 1)

            # Обновляем tqdm: показываем Loss, Acc и текущий LR
            current_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({
                'loss': f"{avg_loss:.4f}",
                'acc': f"{acc:.4f}",
                'lr': f"{current_lr:.6f}"
            })

            # логирование
            num_iter += 1
            if writer is not None:
                writer.add_scalar("Loss/train", loss.item(), num_iter)
                writer.add_scalar("Accuracy/train", correct / batch_size, num_iter)

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        # Валидация (тоже с tqdm)
        val_acc, val_loss, val_f1 = evaluate(model, val_loader, loss_fn_val, device, desc=f"Val {epoch}/{n_epoch}")

        if writer is not None:
            writer.add_scalar("Loss/val", val_loss, num_iter)
            writer.add_scalar("Accuracy/val", val_acc, num_iter)
            writer.add_scalar(f"F1/val", val_f1, num_iter)
            writer.add_scalar(f"LR", current_lr, epoch)

        print(f"Epoch {epoch}/{n_epoch}: val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

        # Проверка на лучшую модель
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_loss = val_loss
            counter_early_stop = 0

            torch.save(model.state_dict(), save_path)
            print(f"--- Эпоха {epoch}: Новая лучшая точность f1: {val_f1:.4f}! Модель сохранена. ---")
        elif (val_f1 == best_val_f1) and (val_loss < best_val_loss):
            best_val_loss = val_loss
            counter_early_stop = 0

            torch.save(model.state_dict(), save_path)
            print(f"--- Эпоха {epoch}: Новая лучшая точность f1: {val_f1:.4f}! Модель сохранена. ---")
        else: # Early stop
            counter_early_stop += 1
            if counter_early_stop >= 5:
                print(f"Сработал Earle stop!")
                break

    # Загружаем лучший вес
    model.load_state_dict(torch.load(f"checkpoints/best_model.pth"))
    os.remove(f"checkpoints/best_model.pth")
    return model

# Итоговая метрика

In [ ]:
@torch.no_grad()
def sklearn_report(model, dataloader, device, idx2class=None, digits=4):
    model.eval()

    y_true, y_pred = [], []

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device, non_blocking=True)

        logits = model(X_batch)
        preds = logits.argmax(dim=1).cpu().numpy()

        y_pred.append(preds)
        y_true.append(y_batch.numpy())

    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)

    # names for report
    if idx2class is None:
        target_names = None
        labels = None
    else:
        labels = sorted(idx2class.keys())
        target_names = [idx2class[i] for i in labels]

    rep = classification_report(
        y_true, y_pred,
        labels=labels,
        target_names=target_names,
        digits=digits,
        zero_division=0
    )
    print(rep)

# Определение весов для классов


In [ ]:
def class_weights(data_counts, num_classes, device):
    total_samples = sum(data_counts.values())
    weights = []

    for i in range(num_classes):
        count = data_counts[i]
        weight = total_samples / (num_classes * count)
        weights.append(weight)

    return torch.FloatTensor(weights).to(device)

class_weights = class_weights(data_counts, num_classes=15, device=device)
print("Веса классов:\n", class_weights)

# Параметры обучения

In [ ]:
os.makedirs('checkpoints', exist_ok=True)
writer = SummaryWriter("logs")

n_fold = 6
ls = 0.1
skf = StratifiedKFold(n_splits=n_fold, shuffle=True, random_state=SEED)
# loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, label_smoothing=ls)
loss_fn = SoftTargetCrossEntropy()

epoch = 25
b_size = 16
lr_ = 3e-4
lr_eta = 1e-6
model_name = 'tf_efficientnetv2_s.in21k_ft_in1k' #'tf_efficientnetv2_s'
data_config = timm.data.resolve_model_data_config(timm.create_model(model_name, pretrained=True))
print(data_config)

# Обучение

In [ ]:
FOLDS_TO_TRAIN = [0, 1, 2, 3, 4, 5]
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_files, all_labels)):
    if fold_idx not in FOLDS_TO_TRAIN:
        continue
    print(f"\n--- Запуск фолда {fold_idx + 1}/{n_fold}  ---")

    # Выбираем файлы по индексам, которые дал SKF
    train_files = all_files[train_idx]
    val_files = all_files[val_idx]

    train_dataset = MyDataset(images_filepaths=train_files, name2label=class_to_idx, transform=train_transforms)
    val_dataset = MyDataset(images_filepaths=val_files, name2label=class_to_idx, transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=b_size, shuffle=True, num_workers=2, pin_memory=True, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=15,
        drop_rate=0.2,
        drop_path_rate=0.1
    )
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr_, weight_decay=1e-2)

    model = train(model, loss_fn, optimizer, train_loader, val_loader,
                  device, writer=None, n_epoch=epoch, lr_eta=lr_eta, weights=class_weights)

    sklearn_report(model, val_loader, device, idx2class={v: k for k, v in class_to_idx.items()})

    # Сохраняем модель текущего фолда
    save_path = f"checkpoints/{model_name}_{fold_idx+1}.pth"
    torch.save(model.state_dict(), save_path)

    # Очистка памяти GPU перед следующим фолдом
    del model, optimizer, train_loader, val_loader
    torch.cuda.empty_cache()

# Предсказание

In [ ]:
w_f1 = [0.9756, 0.9731, 0.9655, 0.9708, 0.9718, 0.9759] # использую значения f1
s = sum(w_f1)
model_weights = [w/s for w in w_f1]
print(model_weights)

In [ ]:
# Веса для моделей
weights_tensor = torch.tensor(model_weights, device=device).view(1, -1, 1)
submission = pd.read_csv(submission_path)

In [ ]:
models = []
# Загрузка моделей
for n in range(6):
    path = f"checkpoints/{model_name}_{n+1}.pth"
    model_i = timm.create_model(model_name, pretrained=False, num_classes=15)
    model_i.load_state_dict(torch.load(path, map_location=device))
    model_i.to(device)
    model_i.eval()
    models.append(model_i)

In [ ]:
final_predictions = []
with torch.no_grad():
    for image_id in tqdm(submission["image_id"], desc="Predicting"):

        image_path = os.path.join(test_images_dir, image_id)
        image = cv2.imdecode(np.fromfile(image_path, dtype=np.uint8), cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if val_transforms is not None:
            image = val_transforms(image=image)["image"]

        img = image.to(device)

        # TTA (Test Time Augmentation)
        img_orig = img
        img_h = torch.flip(img, [2])      # Горизонтально
        img_v = torch.flip(img, [1])      # Вертикально
        img_hv = torch.flip(img, [1, 2])  # И так и так

        # Собираем в батч: [4, 3, 224, 224]
        batch_tta = torch.stack([img_orig, img_h, img_v, img_hv])

        all_models_probs = []

        for m in models:
            logits = m(batch_tta)
            # [4, 15]
            probs_tta = F.softmax(logits, dim=1)
            # [15]
            avg_prob_model = probs_tta.mean(dim=0)
            all_models_probs.append(avg_prob_model)

        # [N_MODEL, 15]
        stacked_probs = torch.stack(all_models_probs)

        weighted_probs = torch.zeros_like(stacked_probs[0]) # [15]

        for i, prob in enumerate(all_models_probs):
            weighted_probs += prob * model_weights[i]
        # Финальный выбор класса
        pred_idx = weighted_probs.argmax().item()
        final_predictions.append(pred_idx)

# Сохранение
submission["label"] = final_predictions
submission.to_csv(output_path, index=False)
submission.head()